In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)


import numpy as np
import torch
import torch.nn as nn
from hedging.envs import HedgeCallBS
from hedging.reward_utils import compute_discounted_cumsum_rewards
from hedging.logit_normal import LogitNormal


from torchrl.envs import GymWrapper
from torchrl.envs.utils import ExplorationType, set_exploration_type
from torchrl.modules import ProbabilisticActor, ValueOperator, ActorValueOperator
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE
from torchrl.collectors import SyncDataCollector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.data import Bounded
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.modules import TanhNormal, ValueOperator, ActorValueOperator, SafeModule


import torch.nn as nn
from tensordict.nn import TensorDictModule, TensorDictSequential, NormalParamExtractor
from tensordict import TensorDict


In [2]:
S0 = np.array([50.0, 100.0, 200.0])
K = np.array([[45.0, 55.0], [90.0, 110.0], [180.0, 220.0]])
maturity = 1.0
r = 0.05
sigma = np.array([0.15, 0.2, 0.25])
num_paths = 100
num_steps = 250
history_len = 1
feature_dim = 11
hidden_dim = 64
action_dim = 1 
transaction_cost = False
transaction_fee_rate = 1e-3

# Param for PPO
clip_param = 0.2
value_coef = 0.1
entropy_coef = 0.001
# Param for GAE
gamma = 0.99
lmbda = 0.95

base_env = HedgeCallBS(
    S0, K, maturity, r, sigma, num_paths, num_steps,
    history_len=history_len,
    transaction_cost=transaction_cost,
    transaction_fee_rate=transaction_fee_rate
)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
env = GymWrapper(base_env, device=device)
env.reset(seed=0)


TensorDict(
    fields={
        done: Tensor(shape=torch.Size([600, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        observation: Tensor(shape=torch.Size([600, 1, 11]), device=cpu, dtype=torch.float32, is_shared=False),
        terminated: Tensor(shape=torch.Size([600, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        truncated: Tensor(shape=torch.Size([600, 1]), device=cpu, dtype=torch.bool, is_shared=False)},
    batch_size=torch.Size([600]),
    device=cpu,
    is_shared=False)

In [ ]:
def generate_actor_inactor_mlp(feat_dim, hidden, action_dim, env, device):
    class FeatureExtractor(nn.Module):
        def __init__(self):
            super(FeatureExtractor, self).__init__()
            self.rnn = nn.LSTM(
                input_size=feat_dim, hidden_size=hidden, num_layers=2, batch_first=True, dropout=0.0
            )

        def forward(self, x):
            if len(x.shape) > 3:  # Handle 4D input
                x_reshaped = x.view(-1, x.shape[-2], x.shape[-1])
                output, _ = self.rnn(x_reshaped)
                # Reshape output back to original batch dimensions
                output = output.view(
                    x.shape[0], x.shape[1], x.shape[2], output.shape[-1]
                )
            else:  # Handle 3D input directly
                output, _ = self.rnn(x)
            output = output[..., -1, :]  # Take output from the last time step
            return output

    actor_feature_extractor = SafeModule(
        module=FeatureExtractor(),
        in_keys=["observation"],
        out_keys=["a_feature"],
    )

    inactor_feature_extractor = SafeModule(
        module=FeatureExtractor(),
        in_keys=["observation"],
        out_keys=["i_feature"],
    )

    actor_policy_network = TensorDictModule(
        nn.Sequential(torch.nn.Linear(64, 2*action_dim), NormalParamExtractor()),
        in_keys=["a_feature"],
        out_keys=["loc", "scale"],
    )

    inactor_policy_network = TensorDictModule(
        nn.Sequential(torch.nn.Linear(64, action_dim)),
        in_keys=["i_feature"],
        out_keys=["logits"],
    )

    action_spec = Bounded(
        low=0.0,
        high=1.0,
        shape=(action_dim,),
        dtype=torch.float,
        device=device,
    )

    actor = ProbabilisticActor(
        module=actor_policy_network,
        in_keys=["loc", "scale"],
        out_keys=["action"],
        distribution_class=TanhNormal,
        return_log_prob=True,
        spec=action_spec,
    )

    inactor = ProbabilisticActor(
        module=inactor_policy_network,
        in_keys=["logits"],
        out_keys=["inact"],
        distribution_class=torch.distributions.Bernoulli,
        return_log_prob=True,
    )

    actor_critic = ValueOperator(
        module=nn.Sequential(torch.nn.Linear(64, 8), nn.Tanh(), nn.Linear(8, 1)),
        in_keys=["a_feature"],
        out_keys=["a_state_value"],
    )

    inactor_critic = ValueOperator(
        module=nn.Sequential(torch.nn.Linear(64, 8), nn.Tanh(), nn.Linear(8, 1)),
        in_keys=["i_feature"],
        out_keys=["i_state_value"],
    )


    actor_model = ActorValueOperator(actor_feature_extractor, actor, actor_critic)
    inactor_model = ActorValueOperator(inactor_feature_extractor, inactor, inactor_critic)

    return actor_model, inactor_model

In [4]:
action_model, inaction_model = generate_actor_inactor_mlp(
    feat_dim=feature_dim,
    hidden=hidden_dim,
    action_dim=action_dim,
    env=env,
    device=device,
)

In [5]:

action_model.to(device)
inaction_model.to(device)

actor_advantage_module = GAE(
    gamma=gamma,
    lmbda=lmbda,
    value_network=action_model.get_value_operator(),
    shifted=True # make sure use this one for RNN
)
actor_advantage_module.set_keys(value="a_state_value")

actor_loss_module = ClipPPOLoss(
    actor_network=action_model.get_policy_operator(),
    critic_network=action_model.get_value_operator(),
    clip_epsilon=clip_param,
    entropy_coef=entropy_coef,
    value_coef=value_coef,
)

actor_loss_module.set_keys(
    action="action",
    sample_log_prob="action_log_prob",
    value="a_state_value"
)

inactor_advantage_module = GAE(
    gamma=gamma,
    lmbda=lmbda,
    value_network=inaction_model.get_value_operator(),
    shifted=True # make sure use this one for RNN
)

inactor_loss_module = ClipPPOLoss(
    actor_network=inaction_model.get_policy_operator(),
    critic_network=inaction_model.get_value_operator(),
    clip_epsilon=clip_param,
    entropy_coef=entropy_coef,
    value_coef=value_coef,
)

inactor_loss_module.set_keys(
    action="inact",
    sample_log_prob="inact_log_prob",
    value="i_state_value"
)

lr=5e-4
optim_actor = torch.optim.Adam(actor_loss_module.parameters(), lr=lr)
optim_inactor = torch.optim.Adam(inactor_loss_module.parameters(), lr=lr)

num_epochs = 10
num_episodes = 100
policy_only_epochs = 5
frames_per_batch = env.num_envs * num_steps
sub_batch_num = 10
sub_batch_size = frames_per_batch // sub_batch_num
frames_per_batch, sub_batch_size

/Users/manu13/Desktop/PHD/DeepHedging/.venv/lib/python3.11/site-packages/torchrl/objectives/ppo.py:511: DeprecationWarning: 'entropy_coef' is deprecated and will be removed in torchrl v0.11. Please use 'entropy_coeff' instead.
  warnings.warn(


(150000, 15000)

In [6]:
with set_exploration_type(ExplorationType.RANDOM):
    for epoch in range(num_epochs):
        for episode in range(num_episodes):
            if epoch < policy_only_epochs:

                env.reset(seed=epoch + 1000)
                collector = SyncDataCollector(
                    env,
                    action_model.get_policy_operator(),
                    frames_per_batch=frames_per_batch,
                    total_frames=frames_per_batch,
                    device=device,
                )
                replay_buffer = ReplayBuffer(
                    storage=LazyTensorStorage(max_size=frames_per_batch),
                    sampler=SamplerWithoutReplacement(),
                )
                for batch in collector:
                    actor_advantage_module(batch)
                    replay_buffer.extend(batch.reshape(-1).cpu())
                    for _ in range(sub_batch_num):
                        subdata = replay_buffer.sample(sub_batch_size)
                        optim_actor.zero_grad()
                        # Forward pass PPO loss
                        loss = actor_loss_module(subdata.to(device))
                        loss_critic, loss_objective, loss_entropy = (
                            loss["loss_critic"],
                            loss["loss_objective"],
                            loss["loss_entropy"],
                        )
                        loss_sum = loss_critic + loss_objective + loss_entropy
                        # Backward pass
                        loss_sum.backward()
                        torch.nn.utils.clip_grad_norm_(actor_loss_module.parameters(), max_norm=1.0)
                        for param in actor_loss_module.parameters():
                            if param.grad is not None:
                                param.grad = torch.nan_to_num(param.grad)
                        # Update the networks
                        optim_actor.step()

                if (episode + 1) % 10 == 0:
                    print(
                        f"""Epoch {epoch+1}/{num_epochs}, Episode {episode + 1}/{num_episodes}, Loss: {loss_sum.item()}, Loss Critic: {loss_critic.item()}, Loss Obj. {loss_objective.item()}, Loss Ent. {loss_entropy.item()}, Avg. Reward: {batch['next', 'reward'].mean().item()}"""
                    )

            else:
                
                actor_advantage_module.set_keys(value="a_state_value")
                inactor_advantage_module.set_keys(value="i_state_value")

                # JointPolicy used for collection
                class JointPolicy(nn.Module):
                    def __init__(self, actor_policy, inactor_policy, action_dim, device):
                        super().__init__()
                        self.actor_policy = actor_policy
                        self.inactor_policy = inactor_policy
                        self.action_dim = action_dim
                        self.device = device
                        self.prev_action = None

                    def forward(self, td):
                        # Avoid mutating collector buffers in-place
                        td = td.clone(False)

                        # Initialize prev_action on first call or batch-size change
                        if (self.prev_action is None) or (self.prev_action.shape[0] != td.shape[0]):
                            self.prev_action = torch.zeros(td.shape[0], self.action_dim, device=self.device)

                        # Run both policies
                        td = self.actor_policy(td)      
                        td = self.inactor_policy(td)   

                        # Persist the actor's original action for its PPO update later
                        td = td.set("original_action", td["action"].clone())

                        # Build inaction mask and mix with previous action
                        inact_val = td["inact"]
                        if inact_val.dim() == 3:
                            mask = inact_val.squeeze(-1).squeeze(-1).bool()
                        elif inact_val.dim() == 2:
                            mask = inact_val.squeeze(-1).bool()
                        else:
                            mask = inact_val.bool()

                        mask = mask.unsqueeze(-1).expand_as(td["action"])
                        final_action = torch.where(mask, self.prev_action, td["action"])
                        td = td.set("action", final_action)
                        self.prev_action = final_action.detach()
                        return td

                    def reset(self):
                        self.prev_action = None

                joint_policy = JointPolicy(
                    action_model.get_policy_operator(),
                    inaction_model.get_policy_operator(),
                    action_dim,
                    device,
                )

                
                collector = SyncDataCollector(
                    env,
                    joint_policy,
                    frames_per_batch=frames_per_batch,
                    total_frames=frames_per_batch,
                    device=device,
                )

                
                replay_buffer_actor = ReplayBuffer(
                    storage=LazyTensorStorage(max_size=frames_per_batch),
                    sampler=SamplerWithoutReplacement(),
                )
                replay_buffer_inactor = ReplayBuffer(
                    storage=LazyTensorStorage(max_size=frames_per_batch),
                    sampler=SamplerWithoutReplacement(),
                )

                for batch in collector:
                    
                    batch_actor = batch.clone(False)
                    batch_inactor = batch.clone(False)

                    if "original_action" in batch_actor.keys():
                        batch_actor.set_("action", batch_actor["original_action"])
                    
                    actor_advantage_module(batch_actor)
                    inactor_advantage_module(batch_inactor)

                    replay_buffer_actor.extend(batch_actor.reshape(-1).cpu())
                    replay_buffer_inactor.extend(batch_inactor.reshape(-1).cpu())

                    # Inaction Policy is Trained
                    for _ in range(sub_batch_num):
                        subdata = replay_buffer_inactor.sample(sub_batch_size).to(device)
                        optim_inactor.zero_grad()
                        loss = inactor_loss_module(subdata)
                        loss_critic = loss["loss_critic"]
                        loss_objective = loss["loss_objective"]
                        loss_entropy = loss["loss_entropy"]
                        loss_sum = loss_critic + loss_objective + loss_entropy
                        loss_sum.backward()
                        torch.nn.utils.clip_grad_norm_(inactor_loss_module.parameters(), max_norm=1.0)
                        for p in inactor_loss_module.parameters():
                            if p.grad is not None:
                                p.grad = torch.nan_to_num(p.grad)
                        optim_inactor.step()
                    inact_loss = loss_sum.item()

                    # Actor Policy Trained Every 3 Losses
                    if episode % 3 == 0:
                        for _ in range(sub_batch_num):
                            subdata = replay_buffer_actor.sample(sub_batch_size).to(device)
                            optim_actor.zero_grad()
                            loss = actor_loss_module(subdata)
                            loss_critic = loss["loss_critic"]
                            loss_objective = loss["loss_objective"]
                            loss_entropy = loss["loss_entropy"]
                            loss_sum = loss_critic + loss_objective + loss_entropy
                            loss_sum.backward()
                            torch.nn.utils.clip_grad_norm_(actor_loss_module.parameters(), max_norm=1.0)
                            for p in actor_loss_module.parameters():
                                if p.grad is not None:
                                    p.grad = torch.nan_to_num(p.grad)
                            optim_actor.step()
                        current_policy_loss = loss_sum.item()
                    else:
                        current_policy_loss = 0.0

                # be nice and reset stateful policy
                joint_policy.reset()

                if (episode + 1) % 10 == 0:
                    policy_info = f"Policy Loss: {current_policy_loss:.4f}" if episode % 3 == 0 else "Policy: No Update"
                    avg_reward = batch["next", "reward"].mean().item()
                    print(
                        f"  Episode {episode + 1}/{num_episodes}, "
                        f"{policy_info}, "
                        f"Inaction Loss: {inact_loss:.4f}, "
                        f"Avg. Reward: {avg_reward:.4f}"
                    )


Epoch 1/10, Episode 10/100, Loss: 362.5259704589844, Loss Critic: 180.97695922851562, Loss Obj. 181.54965209960938, Loss Ent. -0.0006302888505160809, Avg. Reward: -11.10413646697998
Epoch 1/10, Episode 20/100, Loss: 295.7041931152344, Loss Critic: 147.54739379882812, Loss Obj. 148.1571044921875, Loss Ent. -0.00029419618658721447, Avg. Reward: -9.07968807220459
Epoch 1/10, Episode 30/100, Loss: 213.56295776367188, Loss Critic: 106.48681640625, Loss Obj. 107.07573699951172, Loss Ent. 0.00039774031029082835, Avg. Reward: -6.511630058288574
Epoch 1/10, Episode 40/100, Loss: 198.39169311523438, Loss Critic: 98.96235656738281, Loss Obj. 99.4288330078125, Loss Ent. 0.0005099397385492921, Avg. Reward: -6.08400821685791
Epoch 1/10, Episode 50/100, Loss: 193.13162231445312, Loss Critic: 96.33961486816406, Loss Obj. 96.79129791259766, Loss Ent. 0.0007059492636471987, Avg. Reward: -5.917962074279785
Epoch 1/10, Episode 60/100, Loss: 183.40255737304688, Loss Critic: 91.423583984375, Loss Obj. 91.97

KeyboardInterrupt: 

In [7]:
from hedging.plot_utils import plot_portfolio_vs_option_price

In [11]:
# Test

base_env = HedgeCallBS(
    S0, K, maturity, r, sigma, 5, num_steps,
    history_len=history_len,
    transaction_cost=transaction_cost,
    transaction_fee_rate=transaction_fee_rate
)
env = GymWrapper(base_env, device=device)
env.reset(seed=0)

with set_exploration_type(ExplorationType.RANDOM):
    rollout = env.rollout(max_steps=num_steps, policy=joint_policy)

In [12]:
rewards = rollout['next', 'reward'].detach().cpu().numpy()
rewards.min(), rewards.max(), rewards.mean(), rewards.std()

(np.float32(-46.04819),
 np.float32(-0.00012918726),
 np.float32(-5.0848637),
 np.float32(6.3212776))

In [13]:
plot_portfolio_vs_option_price(env._env)